# Day 1 Data Ingestion

Profiles the 10 raw Bluestock datasets, validates AMFI codes, and fetches live NAV files.

The implementation below is embedded from the project scripts so the notebook can be reviewed directly. Run from the project root or keep the notebook in the `notebooks/` folder.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## `data_ingestion.py`

In [ ]:
"""Day 1: ingest and validate the raw Bluestock mutual fund datasets."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

from scripts.project_paths import DATA_PROCESSED, DATA_RAW, ensure_directories


EXPECTED_DATASETS = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv",
]

In [ ]:
def load_raw_dataset(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


def profile_dataframe(name: str, df: pd.DataFrame) -> str:
    lines = [
        f"## {name}",
        "",
        f"- Rows: {len(df):,}",
        f"- Columns: {len(df.columns):,}",
        f"- Duplicate rows: {df.duplicated().sum():,}",
        "- Column dtypes:",
    ]
    for column, dtype in df.dtypes.items():
        null_count = int(df[column].isna().sum())
        lines.append(f"  - `{column}`: {dtype}, nulls={null_count:,}")
    lines.append("")
    lines.append("Sample:")
    lines.append("")
    lines.append(df.head(5).to_markdown(index=False))
    lines.append("")
    return "\n".join(lines)


def validate_amfi_codes(fund_master: pd.DataFrame, nav_history: pd.DataFrame) -> pd.DataFrame:
    master_codes = pd.Series(fund_master["amfi_code"].astype(str).unique(), name="amfi_code")
    nav_codes = set(nav_history["amfi_code"].astype(str).unique())
    return pd.DataFrame(
        {
            "amfi_code": master_codes,
            "exists_in_nav_history": master_codes.isin(nav_codes),
        }
    )

In [ ]:
def main() -> None:
    ensure_directories()
    report_parts = ["# Day 1 Data Ingestion Report", ""]
    loaded: dict[str, pd.DataFrame] = {}

    missing = [name for name in EXPECTED_DATASETS if not (DATA_RAW / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing required raw datasets: {missing}")

    for name in EXPECTED_DATASETS:
        df = load_raw_dataset(DATA_RAW / name)
        loaded[name] = df
        report_parts.append(profile_dataframe(name, df))

    fund_master = loaded["01_fund_master.csv"]
    nav_history = loaded["02_nav_history.csv"]
    code_validation = validate_amfi_codes(fund_master, nav_history)
    code_validation.to_csv(DATA_PROCESSED / "amfi_code_validation.csv", index=False)

    report_parts.extend(
        [
            "## Fund Master Summary",
            "",
            f"- Unique fund houses: {fund_master['fund_house'].nunique():,}",
            f"- Categories: {', '.join(sorted(fund_master['category'].dropna().unique()))}",
            f"- Sub-categories: {', '.join(sorted(fund_master['sub_category'].dropna().unique()))}",
            f"- Risk categories: {', '.join(sorted(fund_master['risk_category'].dropna().unique()))}",
            "",
            "## AMFI Code Validation",
            "",
            f"- Codes in fund master: {len(code_validation):,}",
            f"- Codes found in NAV history: {int(code_validation['exists_in_nav_history'].sum()):,}",
            f"- Missing from NAV history: {int((~code_validation['exists_in_nav_history']).sum()):,}",
            "",
        ]
    )

    (DATA_PROCESSED / "data_quality_report.md").write_text(
        "\n".join(report_parts), encoding="utf-8"
    )
    print("Day 1 ingestion complete.")
    print(f"Report: {DATA_PROCESSED / 'data_quality_report.md'}")

In [ ]:
if __name__ == "__main__":
    main()

## `live_nav_fetch.py`

In [ ]:
"""Day 1: fetch live historical NAV data from mfapi.in for selected schemes."""

from __future__ import annotations

import time
from typing import Iterable

import pandas as pd
import requests

from scripts.project_paths import DATA_RAW, ensure_directories


SCHEMES = {
    "hdfc_top_100": "125497",
    "sbi_bluechip": "119551",
    "icici_bluechip": "120503",
    "nippon_large_cap": "118632",
    "axis_bluechip": "119092",
    "kotak_bluechip": "120841",
}


def fetch_scheme_nav(scheme_code: str) -> pd.DataFrame:
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    payload = response.json()
    rows = payload.get("data", [])
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["amfi_code", "date", "nav"])
    df["amfi_code"] = scheme_code
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
    df["nav"] = pd.to_numeric(df["nav"], errors="coerce")
    df = df.dropna(subset=["date", "nav"]).sort_values("date")
    return df[["amfi_code", "date", "nav"]]

In [ ]:
def fetch_all(schemes: dict[str, str]) -> Iterable[tuple[str, pd.DataFrame]]:
    for slug, code in schemes.items():
        yield slug, fetch_scheme_nav(code)
        time.sleep(0.5)


def main() -> None:
    ensure_directories()
    output_dir = DATA_RAW / "live_nav"
    output_dir.mkdir(parents=True, exist_ok=True)
    for slug, df in fetch_all(SCHEMES):
        output_file = output_dir / f"{slug}_nav.csv"
        df.to_csv(output_file, index=False)
        print(f"Saved {len(df):,} rows: {output_file}")


if __name__ == "__main__":
    main()